In [2]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [4]:
df = pd.read_csv('covid_toy.csv')

In [6]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [7]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [8]:
from sklearn.model_selection import train_test_split as tts
X_train, X_test, y_train, y_test = tts(df.drop(columns=['has_covid']),df['has_covid'], test_size=0.2)

In [9]:
X_train

,age,gender,fever,cough,city
14,51,Male,104.0,Mild,Bangalore
42,27,Male,100.0,Mild,Delhi
46,19,Female,101.0,Mild,Mumbai
17,40,Female,98.0,Strong,Delhi
49,44,Male,104.0,Mild,Mumbai
...,...,...,...,...,...
4,65,Female,101.0,Mild,Mumbai
72,83,Female,101.0,Mild,Kolkata
77,8,Female,101.0,Mild,Kolkata
13,64,Male,102.0,Mild,Bangalore


In [26]:
#In default SimpleImputer replaces null data with mean(default)
si = SimpleImputer(strategy='mean')
X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.fit_transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [11]:
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [16]:
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [27]:
#Extracting Age
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values

X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_train_age.shape


(80, 1)

In [40]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape


(80, 7)

Now with Column Transformer

In [30]:
from sklearn.compose import ColumnTransformer

In [34]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [35]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(),['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]),['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'),['gender','city'])
], remainder='passthrough')

In [37]:
transformer.fit_transform(X_train).shape

(80, 7)

In [41]:
transformer.fit_transform(X_test).shape

(20, 7)